In [0]:
# Set catalog and schema

spark.sql("USE CATALOG final_assesment")
spark.sql("USE SCHEMA gold")

# Load Gold fact and dimension tables

df_fact = spark.table("final_assesment.gold.fact_orders")
df_customer = spark.table("final_assesment.gold.dim_customer")
df_product = spark.table("final_assesment.gold.dim_product")
df_payment = spark.table("final_assesment.gold.dim_payment")
df_date = spark.table("final_assesment.gold.dim_date")

print("Gold tables loaded successfully!")

print("Fact Orders:", df_fact.count())
print("Customers:", df_customer.count())
print("Products:", df_product.count())
print("Payments:", df_payment.count())
print("Dates:", df_date.count())

### Aggrigation 1: Sales by City

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    countDistinct
)

df_sales_by_city = (
    df_fact.alias("f")

    .join(
        df_customer.alias("c"),
        col("f.customer_key") == col("c.customer_key"),
        "inner"
    )

    .groupBy(
        col("c.customer_city")
    )

    .agg(
        countDistinct("f.order_id").alias("total_orders"),

        sum("f.quantity").alias("total_quantity"),

        sum("f.gross_amount").alias("total_gross_sales"),

        sum("f.discount").alias("total_discount"),

        sum("f.net_amount").alias("total_net_sales")
    )

    .orderBy(col("total_net_sales").desc())
)

display(df_sales_by_city)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS final_assesment.gold.agg_sales_by_city (
    customer_city STRING,
    total_orders BIGINT,
    total_quantity BIGINT,
    total_gross_sales DECIMAL(20,2),
    total_discount DECIMAL(20,2),
    total_net_sales DECIMAL(20,2)
)
USING DELTA;

In [0]:
df_sales_by_city.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_assesment.gold.agg_sales_by_city")

print("Sales by City aggregation created successfully!")

In [0]:
%sql

SELECT *
FROM final_assesment.gold.agg_sales_by_city
ORDER BY total_net_sales DESC;

### Aggregation 2: Monthly Sales

In [0]:
df_monthly_sales = (
    df_fact.alias("f")

    .join(
        df_date.alias("d"),
        col("f.date_key") == col("d.date_key"),
        "inner"
    )

    .groupBy(
        col("d.year"),
        col("d.quarter"),
        col("d.month"),
        col("d.month_name")
    )

    .agg(
        countDistinct("f.order_id").alias("total_orders"),

        sum("f.quantity").alias("total_quantity"),

        sum("f.gross_amount").alias("total_gross_sales"),

        sum("f.discount").alias("total_discount"),

        sum("f.net_amount").alias("total_net_sales")
    )

    .orderBy("year", "month")
)

display(df_monthly_sales)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS final_assesment.gold.agg_monthly_sales (
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    total_orders BIGINT,
    total_quantity BIGINT,
    total_gross_sales DECIMAL(20,2),
    total_discount DECIMAL(20,2),
    total_net_sales DECIMAL(20,2)
)
USING DELTA;

In [0]:
df_monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_assesment.gold.agg_monthly_sales")

print("Monthly Sales aggregation created successfully!")

In [0]:
%sql

SELECT *
FROM final_assesment.gold.agg_monthly_sales
ORDER BY year, month;

### Aggregation 3: Payment Analysis

In [0]:
df_payment_analysis = (
    df_fact.alias("f")

    .join(
        df_payment.alias("p"),
        col("f.payment_key") == col("p.payment_key"),
        "inner"
    )

    .groupBy(
        col("p.payment_method"),
        col("p.payment_provider")
    )

    .agg(
        countDistinct("f.order_id").alias("total_orders"),

        sum("f.quantity").alias("total_quantity"),

        sum("f.gross_amount").alias("total_gross_sales"),

        sum("f.discount").alias("total_discount"),

        sum("f.net_amount").alias("total_net_sales")
    )

    .orderBy(col("total_orders").desc())
)

display(df_payment_analysis)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS final_assesment.gold.agg_payment_analysis (
    payment_method STRING,
    payment_provider STRING,
    total_orders BIGINT,
    total_quantity BIGINT,
    total_gross_sales DECIMAL(20,2),
    total_discount DECIMAL(20,2),
    total_net_sales DECIMAL(20,2)
)
USING DELTA;

In [0]:
df_payment_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_assesment.gold.agg_payment_analysis")

print("Payment Analysis aggregation created successfully!")

In [0]:
%sql

SELECT *
FROM final_assesment.gold.agg_payment_analysis
ORDER BY total_orders DESC;

### Aggregation 4: Sales by Category

In [0]:
df_sales_by_category = (
    df_fact.alias("f")

    .join(
        df_product.alias("p"),
        col("f.product_key") == col("p.product_key"),
        "inner"
    )

    .groupBy(
        col("p.category")
    )

    .agg(
        countDistinct("f.order_id").alias("total_orders"),

        sum("f.quantity").alias("total_quantity"),

        sum("f.gross_amount").alias("total_gross_sales"),

        sum("f.discount").alias("total_discount"),

        sum("f.net_amount").alias("total_net_sales")
    )

    .orderBy(col("total_net_sales").desc())
)

display(df_sales_by_category)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS final_assesment.gold.agg_sales_by_category (
    category STRING,
    total_orders BIGINT,
    total_quantity BIGINT,
    total_gross_sales DECIMAL(20,2),
    total_discount DECIMAL(20,2),
    total_net_sales DECIMAL(20,2)
)
USING DELTA;

In [0]:
df_sales_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_assesment.gold.agg_sales_by_category")

print("Sales by Category aggregation created successfully!")

In [0]:
%sql

SELECT *
FROM final_assesment.gold.agg_sales_by_category
ORDER BY total_net_sales DESC;

### Final validation of all four tables

In [0]:
%sql

SHOW TABLES IN final_assesment.gold;

In [0]:
%sql

SELECT
    'Sales by City' AS aggregation,
    SUM(total_orders) AS total_orders,
    SUM(total_net_sales) AS total_net_sales
FROM final_assesment.gold.agg_sales_by_city

UNION ALL

SELECT
    'Monthly Sales',
    SUM(total_orders),
    SUM(total_net_sales)
FROM final_assesment.gold.agg_monthly_sales

UNION ALL

SELECT
    'Payment Analysis',
    SUM(total_orders),
    SUM(total_net_sales)
FROM final_assesment.gold.agg_payment_analysis

UNION ALL

SELECT
    'Sales by Category',
    SUM(total_orders),
    SUM(total_net_sales)
FROM final_assesment.gold.agg_sales_by_category;